`conda activate imgpro`

In [42]:
import sys
import os
import importlib
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import shutil
import glob
import utils
import tifffile
import numpy as np
from scipy.io import savemat
import json

In [29]:
DATA_PTH = r'D:\DATA'
PROJECT = 'g5ht-free' # g5ht-free or g5ht-immo
OUT_PTH = 'date-20251028_time-1500_strain-ISg5HT_condition-starvedpatch_worm001'
reg_dir = 'registered_elastix'
# reg_dir = 'registered_wholistic_smooth-0.200_patch-7' # for 20251223 worm005

DATE = OUT_PTH.split('_')[0].split('-')[1] 
DATA_PTH = os.path.join(DATA_PTH, PROJECT, DATE)
pth = os.path.join(DATA_PTH, OUT_PTH)

reg_dir = os.path.join(pth, reg_dir)
reg_tifs = [f for f in os.listdir(reg_dir) if f.endswith('.tif')]
reg_tifs.sort() # ensure correct order
reg_tifs_pths = [os.path.join(reg_dir, f) for f in reg_tifs]

# load mask
mask_fn = glob.glob(os.path.join(pth, 'fixed_mask_[0-9][0-9][0-9][0-9]*.tif'))[0]
mask = tifffile.TiffFile(os.path.join(pth, mask_fn)).asarray()
print(f"Mask shape: {mask.shape}")

# load first frame to get dimensions
with tifffile.TiffFile(reg_tifs_pths[0]) as tif:
    Z,C,H,W = tif.asarray().shape
print(f"Dimensions of registered tifs: Z={Z}, C={C}, H={H}, W={W}")


Mask shape: (200, 500)
Dimensions of registered tifs: Z=40, C=2, H=200, W=500


In [30]:
# in pth, make an optical_flow directory if it doesn't already exist
of_pth = os.path.join(pth, 'optical_flow')
if not os.path.exists(of_pth):
    os.mkdir(of_pth)

In [41]:
data = dict()
data['mip'] = mip

In [40]:
mip = np.full((len(reg_tifs_pths),C,H,W),np.nan)

for i,tif_pth in tqdm(enumerate(reg_tifs_pths)):
    # print(tif_pth)

    with tifffile.TiffFile(reg_tifs_pths[i]) as tif:
        im = np.max(tif.asarray(),axis=0) # CHW

    mip[i,:,:,:] = im

data = dict()
data['mip'] = mip

1200it [01:04, 18.52it/s]


In [47]:
# add metadata
with open(os.path.join(pth,'metadata.json'), 'r') as f:
    metadata = json.load(f)
metadata['bad_frames'] = np.array(metadata['bad_frames'])  # convert back to numpy array
metadata['frame_index'] = np.array(metadata['frame_index'])  # convert back to numpy array
print(metadata)

for key in metadata.keys():
    print(key)
    data[key] = metadata[key]

{'fps': 1.8761726078799248, 'nframes': 1200, 'baseline_start_frame': 0, 'baseline_end_frame': 60, 'encounter_frame': 128, 'bad_frames': array([], dtype=float64), 'frame_index': array([   0,    1,    2, ..., 1197, 1198, 1199], shape=(1200,))}
fps
nframes
baseline_start_frame
baseline_end_frame
encounter_frame
bad_frames
frame_index


In [57]:
# add mask
data['mask'] = mask

In [58]:
fn = INPUT_ND2.split('.')[0] + '_of_data.mat'
savemat(os.path.join(of_pth, fn), data)

In [51]:
of_pth

'D:\\DATA\\g5ht-free\\20251028\\date-20251028_time-1500_strain-ISg5HT_condition-starvedpatch_worm001\\optical_flow'